In [1]:
import os
os.environ["JOBLIB_START_METHOD"] = "forkserver"
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["OPENBLAS_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"

import pandas as pd
import os
import matplotlib.pyplot as plt
import plotly.graph_objects as go
import json
import mne
import numpy as np
from tqdm import tqdm
import re
from sklearn.metrics import silhouette_score, calinski_harabasz_score, davies_bouldin_score
from multiprocessing import Pool
import warnings

warnings.filterwarnings("ignore")

os.chdir('../..')

# Load Morlet PSDs

In [2]:
loaded = np.load('./Generated/Spectrums/exec_morlets.npz')

results_arr = []

i = 0
while f'power_{i}' in loaded:
    
    power = loaded[f'power_{i}']
    phase = loaded[f'phase_{i}']
    s_id = int(loaded[f'subject_id_{i}'])
    t_id = int(loaded[f'trial_id_{i}'])
    gender = str(loaded[f'gender_{i}'])
    handiness = str(loaded[f'handiness_{i}'])
    age = int(loaded[f'age_{i}'])
    label = int(loaded[f'label_{i}'])
    img = loaded[f'img_{i}']
    task_type = str(loaded[f'task_type_{i}'])
    
    results_arr.append([power, phase, s_id, t_id, gender, handiness, age, label, img, task_type])
    i += 1

power, phase, s_id, t_id, gender, handiness, age, label, img, task_type = results_arr[0]
power.shape

(63, 195, 321)

In [3]:
# Шаг 1: Найти минимальный размер по последней оси
min_len = min(power.shape[-1] for power, *_ in results_arr)

# Шаг 2: Обрезать все psd до min_len с двух сторон
new_results_arr = []
for power, phase, s_id, t_id, gender, handiness, age, label, img, task_type in results_arr:
    cur_len = power.shape[-1]
    if cur_len > min_len:
        diff = cur_len - min_len
        left = diff // 2
        right = diff - left
        power = power[:, :, left:cur_len - right]
    new_results_arr.append((power, phase, s_id, t_id, gender, handiness, age, label, img, task_type))

results_arr = new_results_arr

power, phase, s_id, t_id, gender, handiness, age, label, img, task_type = results_arr[0]
power.shape, phase.shape

((63, 195, 309), (63, 195, 321))

In [4]:
# Parse into two lists will be used furthere
psds_array = [
    np.stack([
        item[0][..., :min(item[0].shape[-1], item[1].shape[-1])],  # power
        item[1][..., :min(item[0].shape[-1], item[1].shape[-1])]   # phase
    ], axis=-1)
    for item in results_arr
]

psds_array = np.array(psds_array)                            # Convert list of PSDs to a numpy array
psds_array = psds_array.reshape((psds_array.shape[0], -1))   # Vectorize 

metadata = [item[1:] for item in results_arr]

psds_array.shape

(1260, 7592130)

### Normalize Each Spectrum
Why not to normalize by channels? - Because the chennel relevance to each other may be lost. For instance some chanels may have higher power for particular freq band.

In [5]:
# from sklearn.preprocessing import StandardScaler
# scaler = StandardScaler()
# psds_array = scaler.fit_transform(psds_array)

# Dimentionality Reduction
Every method produces `reduced_data` variable wich is `27xN` shape, where `N` is the reduction dimentionality. 

### UMAP

In [6]:
# from sklearn.manifold import trustworthiness
# import umap

# # Примерные значения параметров для перебора
# n_components_list = range(2, 30)
# n_neighbors_list = [2, 3, 4, 5, 6, 7, 8]
# min_dist_list = [0.1, 0.2, 0.3, 0.5]
# metric_list = ['cosine', 'euclidean']

# scores = []

# for n_components in tqdm(n_components_list):
#     for n_neighbors in n_neighbors_list:
#         for min_dist in min_dist_list:
#             for metric in metric_list:
#                 reducer = umap.UMAP(
#                     n_components=n_components,
#                     n_neighbors=n_neighbors,
#                     min_dist=min_dist,
#                     metric=metric,
#                     random_state=42
#                 )
#                 embedding = reducer.fit_transform(psds_array)
#                 trust = trustworthiness(psds_array, embedding, n_neighbors=n_neighbors, metric=metric)
#                 scores.append((trust, n_components, n_neighbors, min_dist, metric))
#                 # print(f"n_components={n_components}, n_neighbors={n_neighbors}, min_dist={min_dist}, metric={metric}, trustworthiness={trust:.3f}")

# # Сортируем по убыванию trustworthiness
# top_n = 3
# top = sorted(scores, key=lambda x: x[0], reverse=True)[:top_n]

# print(f"Top {top_n} parameter combinations by trustworthiness:")
# for rank, (trust, n_components, n_neighbors, min_dist, metric) in enumerate(top, start=1):
#     print(f"{rank}. trustworthiness={trust:.3f} | n_components={n_components}, n_neighbors={n_neighbors}, min_dist={min_dist}, metric={metric}")

In [ ]:
import umap.umap_ as umap
import joblib

# UMAP for dimensionality reduction
with joblib.parallel_backend('threading', n_jobs=40):
    reducer = umap.UMAP(n_components=20, n_neighbors=3, min_dist=0.4, metric='cosine', random_state=None, n_jobs=40)
    reduced_data = reducer.fit_transform(psds_array)

IOStream.flush timed out


# Clusterization
* `KNN` - 
* `DBSCAN` - 
* `HDBSCAN` -
* `GMM` -
* `Spectral Clustering` -
* `Agglomerative` - 

### KNN
Best for `PCA` k=2 (0.378 оно всеравно меньше 0.5, а значит не столь надежно)

Best for `UMAP` k=3 (0.781 - Достаточно надежно!)

In [ ]:
from sklearn.cluster import KMeans

silhouette_scores = []
calinski_scores = []
davies_scores = []
k_range = range(2, 15)

for k in k_range:
    
    kmeans = KMeans(n_clusters=k, random_state=42)
    labels = kmeans.fit_predict(reduced_data)

    # Метрики
    sil = silhouette_score(reduced_data, labels)
    cal = calinski_harabasz_score(reduced_data, labels)
    dav = davies_bouldin_score(reduced_data, labels)

    silhouette_scores.append(sil)
    calinski_scores.append(cal)
    davies_scores.append(dav)

# Визуализация всех трех метрик
fig, axs = plt.subplots(1, 3, figsize=(14, 4))

axs[0].plot(k_range, silhouette_scores, marker='o')
axs[0].set_title("Silhouette Score (Maximize)")
axs[0].set_ylabel("Score")
axs[0].grid(True)

axs[1].plot(k_range, calinski_scores, marker='o', color='green')
axs[1].set_title("Calinski-Harabasz Score (Maximize)")
axs[1].set_ylabel("Score")
axs[1].grid(True)

axs[2].plot(k_range, davies_scores, marker='o', color='red')
axs[2].set_title("Davies-Bouldin Score (Minimize)")
axs[2].set_xlabel("Number of Clusters")
axs[2].set_ylabel("Score")
axs[2].grid(True)

plt.tight_layout()
plt.show()

In [ ]:
k = 4  # предположительное количество кластеров
kmeans = KMeans(n_clusters=k, random_state=42)
labels_knn = kmeans.fit_predict(reduced_data)

### DBSCAN
Best for `PCA` - Излом на 180

In [ ]:
from sklearn.neighbors import NearestNeighbors

min_samples = 2
neighbors = NearestNeighbors(n_neighbors=min_samples)
neighbors_fit = neighbors.fit(reduced_data)
distances, indices = neighbors_fit.kneighbors(reduced_data)

# берем расстояние до k-го соседа (по столбцу)
k_distances = np.sort(distances[:, -1])

plt.figure(figsize=(6, 4))
plt.plot(k_distances)
plt.title(f"k-distance Graph (k={min_samples})")
plt.xlabel("Points sorted by distance")
plt.ylabel(f"Distance to {min_samples}th nearest neighbor")
plt.grid(True)
plt.show()

In [ ]:
from sklearn.cluster import DBSCAN

# DBSCAN clustering in high-dimensional space
dbscan = DBSCAN(eps=800, min_samples=2)  # eps можно варьировать
labels_dbscan = dbscan.fit_predict(reduced_data)

### HDBSCAN

In [ ]:
import hdbscan

clusterer = hdbscan.HDBSCAN(min_cluster_size=13)
labels_hdbscan = clusterer.fit_predict(reduced_data)

In [ ]:
clusterer.condensed_tree_.plot()
pass

### GMM
`n_components`: количество гауссовых компонентов. Подбирается через BIC:

BIC (Bayesian Information Criterion) — это метрика, которая балансирует:

качество подгонки модели (log-likelihood — чем выше, тем лучше)

сложность модели (штраф за число параметров)


Best for `PCA` - minimum bic is for 8 components (похоже на выброс, лучше использовать 4)

In [ ]:
from sklearn.mixture import GaussianMixture

bic_scores = []
for n in range(1, 15):
    gmm = GaussianMixture(n_components=n, random_state=42)
    gmm.fit(reduced_data)
    bic_scores.append(gmm.bic(reduced_data))

plt.plot(range(1, 15), bic_scores)
plt.title("BIC Scores for GMM")
plt.xlabel("Number of components")
plt.ylabel("BIC")
plt.show()

In [ ]:
silhouette_scores = []
calinski_scores = []
davies_scores = []
k_range = range(2, 15)

for k in k_range:
    
    gmm = GaussianMixture(n_components=k, random_state=42)
    labels = gmm.fit_predict(reduced_data)

    # Метрики
    sil = silhouette_score(reduced_data, labels)
    cal = calinski_harabasz_score(reduced_data, labels)
    dav = davies_bouldin_score(reduced_data, labels)

    silhouette_scores.append(sil)
    calinski_scores.append(cal)
    davies_scores.append(dav)

# Визуализация всех трех метрик
fig, axs = plt.subplots(1, 3, figsize=(14, 4))

axs[0].plot(k_range, silhouette_scores, marker='o')
axs[0].set_title("Silhouette Score (Maximize)")
axs[0].set_ylabel("Score")
axs[0].grid(True)

axs[1].plot(k_range, calinski_scores, marker='o', color='green')
axs[1].set_title("Calinski-Harabasz Score (Maximize)")
axs[1].set_ylabel("Score")
axs[1].grid(True)

axs[2].plot(k_range, davies_scores, marker='o', color='red')
axs[2].set_title("Davies-Bouldin Score (Minimize)")
axs[2].set_xlabel("Number of Clusters")
axs[2].set_ylabel("Score")
axs[2].grid(True)

plt.tight_layout()
plt.show()

In [ ]:
gmm = GaussianMixture(n_components=4, random_state=42)
labels_gmm = gmm.fit_predict(reduced_data)

### Spectral Clustering

Best for `PCA` - 2 clusters

In [ ]:
from sklearn.cluster import SpectralClustering

silhouette_scores = []
calinski_scores = []
davies_scores = []
k_range = range(2, 15)

for k in k_range:
    spec = SpectralClustering(n_clusters=k, affinity='nearest_neighbors', random_state=42)
    labels = spec.fit_predict(psds_array)

    # Метрики
    sil = silhouette_score(reduced_data, labels)
    cal = calinski_harabasz_score(reduced_data, labels)
    dav = davies_bouldin_score(reduced_data, labels)

    silhouette_scores.append(sil)
    calinski_scores.append(cal)
    davies_scores.append(dav)

# Визуализация всех трех метрик
fig, axs = plt.subplots(1, 3, figsize=(14, 4))

axs[0].plot(k_range, silhouette_scores, marker='o')
axs[0].set_title("Silhouette Score (Maximize)")
axs[0].set_ylabel("Score")
axs[0].grid(True)

axs[1].plot(k_range, calinski_scores, marker='o', color='green')
axs[1].set_title("Calinski-Harabasz Score (Maximize)")
axs[1].set_ylabel("Score")
axs[1].grid(True)

axs[2].plot(k_range, davies_scores, marker='o', color='red')
axs[2].set_title("Davies-Bouldin Score (Minimize)")
axs[2].set_xlabel("Number of Clusters")
axs[2].set_ylabel("Score")
axs[2].grid(True)

plt.tight_layout()
plt.show()

In [ ]:
spec = SpectralClustering(n_clusters=4, affinity='nearest_neighbors', random_state=42)
labels_spec_clust = spec.fit_predict(psds_array)

### Agglomerative

Best for `PCA` - 5 clusters (max score)

In [ ]:
from sklearn.cluster import AgglomerativeClustering

silhouette_scores = []
calinski_scores = []
davies_scores = []
k_range = range(2, 15)

for k in k_range:
    model = AgglomerativeClustering(n_clusters=k, linkage='ward')
    labels = model.fit_predict(reduced_data)

    # Метрики
    sil = silhouette_score(reduced_data, labels)
    cal = calinski_harabasz_score(reduced_data, labels)
    dav = davies_bouldin_score(reduced_data, labels)

    silhouette_scores.append(sil)
    calinski_scores.append(cal)
    davies_scores.append(dav)

# Визуализация всех трех метрик
fig, axs = plt.subplots(1, 3, figsize=(14, 4))

axs[0].plot(k_range, silhouette_scores, marker='o')
axs[0].set_title("Silhouette Score (Maximize)")
axs[0].set_ylabel("Score")
axs[0].grid(True)

axs[1].plot(k_range, calinski_scores, marker='o', color='green')
axs[1].set_title("Calinski-Harabasz Score (Maximize)")
axs[1].set_ylabel("Score")
axs[1].grid(True)

axs[2].plot(k_range, davies_scores, marker='o', color='red')
axs[2].set_title("Davies-Bouldin Score (Minimize)")
axs[2].set_xlabel("Number of Clusters")
axs[2].set_ylabel("Score")
axs[2].grid(True)

plt.tight_layout()
plt.show()

In [ ]:
# Иерархическая кластеризация с заданным числом кластеров
model = AgglomerativeClustering(n_clusters=5, linkage='ward')
labels_agglomerative = model.fit_predict(reduced_data)

In [ ]:
from scipy.cluster.hierarchy import dendrogram, linkage

linked = linkage(reduced_data, method='ward')  # метод linkage должен соответствовать
plt.figure(figsize=(10, 5))
dendrogram(linked, truncate_mode='lastp', p=30, leaf_rotation=90.)
plt.title('Дендрограмма')
plt.xlabel('Точки или кластеры')
plt.ylabel('Расстояние')
plt.show()

# Search Space Visualization

In [ ]:
import umap.umap_ as umap

# UMAP for dimensionality reduction
reducer = umap.UMAP(n_components=3, n_neighbors=3, min_dist=0.4, metric='cosine', random_state=42)
reduced_data = reducer.fit_transform(psds_array)

In [ ]:
import plotly.graph_objects as go

method = 'UMAP'
dim = 3  # Can be 2 or 3

subject_ids = [meta[0] for meta in metadata]
trial_ids = [meta[1] for meta in metadata]
gender = [meta[2] for meta in metadata]
handiness = [meta[3] for meta in metadata]
age = [meta[4] for meta in metadata]

true_label = [meta[5] for meta in metadata]
task_type = [meta[7] for meta in metadata]

# Adjust DataFrame columns based on dim
columns = [f'DIM-{i+1}' for i in range(dim)]
df = pd.DataFrame(reduced_data, columns=columns)
df['Subject_ID'] = subject_ids
df['Trial_ID'] = trial_ids
df['Gender'] = gender
df['Handiness'] = handiness
df['Age'] = age
df['True_Labels'] = true_label
df['Task Type'] = task_type 
df['Labels_DBSCAN'] = labels_dbscan
df['Labels_KNN'] = labels_knn
df['Labels_HDBSCAN'] = labels_hdbscan
df['Labels_GMM'] = labels_gmm
df['Labels_Spectral_Clustering'] = labels_spec_clust
df['Labels_Agglomerative'] = labels_agglomerative
df['Metadata'] = [str(meta) for meta in metadata]

coloring_vars = ['Subject_ID', 'Trial_ID', 'Gender', 'Handiness', 'Age', 'True_Labels', 'Task Type', 'Labels_DBSCAN', 'Labels_KNN', 'Labels_HDBSCAN', 'Labels_GMM', 'Labels_Spectral_Clustering', 'Labels_Agglomerative']

fig = go.Figure()

for var in coloring_vars:
    if var in ['Subject_ID', 'Trial_ID', 'Gender', 'Handiness', 'Age', 'True_Labels', 'Task Type', 'Labels_DBSCAN', 'Labels_KNN', 'Labels_HDBSCAN', 'Labels_GMM', 'Labels_Spectral_Clustering', 'Labels_Agglomerative']:
        unique_values = df[var].unique()
        color_map = {val: i for i, val in enumerate(unique_values)}
        colors = [color_map[val] for val in df[var]]
    else:
        colors = df[var]

    if dim == 3:
        fig.add_trace(go.Scatter3d(
            x=df['DIM-1'],
            y=df['DIM-2'],
            z=df['DIM-3'],
            mode='markers',
            marker=dict(color=colors),
            hoverinfo='text',
            hovertext=df['Metadata'],
            visible=var == 'Cluster'
        ))
    else:  # dim == 2
        fig.add_trace(go.Scatter(
            x=df['DIM-1'],
            y=df['DIM-2'],
            mode='markers',
            marker=dict(color=colors),
            hoverinfo='text',
            hovertext=df['Metadata'],
            visible=var == 'Cluster'
        ))

buttons = []
for i, var in enumerate(coloring_vars):
    buttons.append(dict(
        label=var,
        method='update',
        args=[{'visible': [j == i for j in range(len(coloring_vars))]}]
    ))

# Update layout based on dimension
if dim == 3:
    fig.update_layout(
        updatemenus=[dict(
            buttons=buttons,
            direction='down',
            pad={'r': 10, 't': 10},
            showactive=True,
            x=0.1,
            xanchor='left',
            y=1.15,
            yanchor='top'
        )],
        title=f'{method} Reduced Space ({dim}D)',
        scene=dict(
            xaxis_title=f'{method}-1',
            yaxis_title=f'{method}-2',
            zaxis_title=f'{method}-3'
        )
    )
elif dim == 2:
    fig.update_layout(
        updatemenus=[dict(
            buttons=buttons,
            direction='down',
            pad={'r': 10, 't': 10},
            showactive=True,
            x=0.1,
            xanchor='left',
            y=1.15,
            yanchor='top'
        )],
        title=f'{method} Reduced Space ({dim}D)',
        xaxis_title=f'{method}-1',
        yaxis_title=f'{method}-2'
    )

fig.show()
# fig.write_html(f'./Generated/Figures/Spectral_Analysis/Spectral_Clustering/All/FFT_PCA_All_Clust_3D.html', include_plotlyjs='cdn', full_html=True)

# Try Predicting Gender
This experiment shows the possibility of gender prediction based only on the clusterization data. The one considers the best possible case of predicting cluster to be `m` or `f` showing the *posibility* to separate spectras by gender.

In [ ]:
from collections import Counter
from sklearn.metrics import classification_report, accuracy_score

gender = np.array(gender)
label_sets = [
    ("HDBSCAN", labels_hdbscan),
    ("GMM", labels_gmm),
    ("Spectral Clustering", labels_spec_clust),
    ("Agglomerative", labels_agglomerative)
]


# Основной цикл
for name, labels in label_sets:
    print(f"\n==================================== {name} ====================================")
    cluster_stats = {}
    for cluster in np.unique(labels):
        g_in_cluster = gender[labels == cluster]
        counts = Counter(g_in_cluster)
        cluster_stats[cluster] = counts
        print(f"Cluster {cluster}: {dict(counts)}")
    
    # Маппинг кластера к полу по большинству
    cluster_to_gender = {
        cluster: counts.most_common(1)[0][0]
        for cluster, counts in cluster_stats.items()
    }

    # Прогнозы
    y_pred = np.array([cluster_to_gender[label] for label in labels])
    y_true = gender

    # Метрики
    print("\nClassification report:")
    print(classification_report(y_true, y_pred))
    accuracy = accuracy_score(y_true, y_pred)
    print(f"Accuracy: {accuracy:.4f}\n")